<a href="https://colab.research.google.com/github/yin-penghang/AMAT593/blob/main/18_Applications_III/01_Collaborative_Filtering_Recommendation_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collaborative Filtering Recommendation Systems

[Collaborative filtering](https://en.wikipedia.org/wiki/Collaborative_filtering) aims to fill in the missing entries of a user-item rating matrix with predicted ratings, so that users are recomended new items based on the predicted ratings.

In this notebook, we will introduce the classical matrix factorization model for collaborative filtering as well as the state-of-the-arts based on neural networks.


<img src="https://github.com/yin-penghang/AMAT593/blob/main/figs/15_CF.png?raw=true" width = '400'>

# Matrix Factorization

Collaborative filtering is an application of matrix factorization to identify the relationship between items’ and users’ entities. With the input of users’ ratings on some of the items, we would like to predict how the users would rate the other items so the users can get the recommendation based on the prediction.

Suppose we have the user-item rating matrix $\mathbf{R}\in\mathbb{R}^{m\times n}$ for $m$ users and $n$ items, and the ratings are integers ranging from 1 to 5:

<img src= 'https://github.com/yin-penghang/AMAT593/blob/main/figs/15_rating_matrix.png?raw=true' width = '800'>

The $(i,j)$-th entry $r_{i,j}$ represents the $i$-th user's rating on the $j$-th item.
The rating matrix $\mathbf{R}\in\mathbb{R}^{m\times n}$ has a large portion of entries/ratings missing, and our goal here is to impute the full matrix.

Matrix factorization model assumes: every item has the same set of $k$ **latent factors** based on which a user will give the rating, where **$k$ is much less than the number of users and the number of items**. Maybe one factor means "movies with frantic chases", another factor might mean "movies with a plot twist", etc.

Let $\mathbf{q}_j \in\mathbb{R}^k$ be the $j$-th item's **components** in the $k$ latent factors and $\mathbf{p}_i\in\mathbb{R}^k$ be the $i$-th user's **personal preferences** on these $k$ latent factors, e.g.,

    Movie j = 0.5 x frantic chases + 0.2 x plot twist + ...
    
    User i = 0.3 x fan of frantic chases + 1.8 x fan of plot twist + ...

In practice, the latent factors are opaque. Given $\mathbf{q}_j$ and $\mathbf{p}_i$, the model generates the rating:

$$
\hat{r}_{i,j} = \mathbf{p}_i^\top \mathbf{q}_j
$$

In matrix form, denote by $\mathbf{P} = \begin{bmatrix} | \qquad | \\ \mathbf{p}_1 \cdots \mathbf{p}_m \\ | \qquad | \end{bmatrix}\in\mathbb{R}^{k\times m}$ the user latent matrix, and $\mathbf{Q} = \begin{bmatrix} | \qquad | \\ \mathbf{q}_1 \cdots \mathbf{q}_n \\ | \qquad | \end{bmatrix}\in\mathbb{R}^{k\times n}$ the item latent matrix, then the predicted rating matrix has the **low-rank matrix factorization**

$$
\hat{\mathbf{R}} = \mathbf{P}^\top \, \mathbf{Q}\in\mathbb{R}^{m\times n}
$$

<img src='https://github.com/yin-penghang/AMAT593/blob/main/figs/15_MatFacto.png?raw=true' width = '500'>

For any observed rating $r_{i,j}\in\mathbf{R}$, $\hat{\mathbf{R}}$ satisfies

$$
\hat{r}_{i,j} = \mathbf{p}_i^\top \mathbf{q}_j \approx r_{i,j}.
$$

**Under the constraints above, we aim to infer the user latent matrix $\mathbf{P}$ and item latent matrix $\mathbf{Q}$** by solving

$$
\min_{\{\mathbf{p}_i\}_{i=1}^m, \{\mathbf{q}_j\}_{j=1}^n} \; \sum_{\mbox{ observed } r_{i,j}} (r_{i,j} - \mathbf{p}_i^\top \mathbf{q}_j)^2
+ \lambda(\|\mathbf{p}_i\|^2 + \|\mathbf{q}_j\|^2)
$$

### The Algorithm

The minimization is performed by a straightforward mini-batch gradient descent: for an observed rating $r_{i,j}$, iterate

\begin{align}
\mathbf{p}_i \leftarrow \mathbf{p}_i - \eta  \left( (\mathbf{p}_i^\top \mathbf{q}_j - r_{i,j})\mathbf{q}_j + \lambda \mathbf{p}_i \right) \\
\mathbf{q}_j \leftarrow \mathbf{q}_j - \eta  \left( (\mathbf{p}_i^\top \mathbf{q}_j - r_{i,j})\mathbf{p}_i + \lambda \mathbf{q}_j \right)
\end{align}

The above iterations are known as the SVD algorithm, as popularized by [Simon Funk](https://sifter.org/~simon/journal/20061211.html) during the Netflix Prize, although
**it is actually NOT [SVD](https://en.wikipedia.org/wiki/Singular_value_decomposition) which computes the factorization of a full matrix.** The more appropriate term for what Funk did is **matrix factorization**.

## Tensorflow Implementation

We implement matrix factorization model using the Tensorflow/Keras libraries. The key for the implementation is to represent the user latent matrix $P\in\mathbb{R}^{k\times m}$ and item latent matrix $Q\in\mathbb{R}^{k\times n}$ with two **Embedding** layers. With this notion, $P$ and $Q$ can be viewed as trainable weight matrices of the model.

In [ ]:
import pandas as pd
import numpy as np
# import random

In [ ]:
import tensorflow as tf
from keras.models import Model, Sequential
from keras import regularizers
from keras.layers import Embedding, Flatten, Input, dot
from tensorflow.keras.utils import model_to_dot
from IPython.display import SVG, HTML

In [ ]:
def MatrixFacto(n_users, n_movies, latent_dim):

    movie_input = Input(shape=[1])
    movie_embedding = Embedding(input_dim = n_movies, output_dim = latent_dim, embeddings_regularizer=regularizers.L2(l2=1e-6))(movie_input)
    movie_vec = Flatten()(movie_embedding)

    user_input = Input(shape=[1])
    user_embedding = Embedding(input_dim = n_users, output_dim = latent_dim, embeddings_regularizer=regularizers.L2(l2=1e-6))(user_input)
    user_vec = Flatten()(user_embedding)

    dot_prod = dot([movie_vec, user_vec], axes=1)


    model = Model(inputs = [user_input, movie_input], outputs = dot_prod)
    model.compile(loss = 'mse', optimizer = 'adam', metrics = [tf.keras.metrics.RootMeanSquaredError(name='rmse')])


    return model

### MovieLens Dataset

MovieLens small dataset contains ~100,000 ratings from ~600 users on ~9700 movies.

In [ ]:
from google.colab import files

# Reading ratings file
print("Please upload the 'ratings.csv' file from your local machine.")
uploaded = files.upload()

# Assuming the uploaded file is named 'ratings.csv'.
# If you uploaded a file with a different name, please adjust 'file_name' below.
file_name = list(uploaded.keys())[0] if uploaded else 'ratings.csv'

data = pd.read_csv(file_name, sep=',', encoding='latin-1', usecols=['userId','movieId','rating'])
data.head(10)

Please upload the 'ratings.csv' file from your local machine.


,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
5,1,70,3.0
6,1,101,5.0
7,1,110,4.0
8,1,151,5.0
9,1,157,5.0


In [ ]:
print(data.movieId.unique())

[     1      3      6 ... 160836 163937 163981]


In [ ]:
print(data.userId.unique())

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 239 240 241 242 243 244 245 246 24

Need to map movie IDs to `[0, n_movies-1]` and user IDs to `[0, n_user -1]`. Note that the movie IDs are not consecutive.

In [ ]:
movie_id_to_new_id = dict()
id = 0
for index, row in data.iterrows():
    if movie_id_to_new_id.get(row['movieId']) is None:
        movie_id_to_new_id[row['movieId']] = id
        data.at[index, 'movieId'] = id
        id += 1
    else:
        data.at[index, 'movieId'] = movie_id_to_new_id.get(row['movieId'])

In [ ]:
data.userId = data.userId-1

In [ ]:
data.head(10)

,userId,movieId,rating
0,0,0,4.0
1,0,1,4.0
2,0,2,4.0
3,0,3,5.0
4,0,4,5.0
5,0,5,3.0
6,0,6,5.0
7,0,7,4.0
8,0,8,5.0
9,0,9,5.0


In [ ]:
n_users = data.userId.unique().shape[0]
n_movies = data.movieId.unique().shape[0]
print('Number of users = ' + str(n_users) + '\nNumber of movies = ' + str(n_movies))

Number of users = 610
Number of movies = 9724


Let's first check the sparsity of the ratings dataset, i.e., the percentage of missing ratings:

In [ ]:
sparsity = round(1.0 - len(data) / float(n_users * n_movies), 3)
print('The sparsity level of MovieLens dataset is ' +  str(sparsity * 100) + '%')

The sparsity level of MovieLens dataset is 98.3%


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state = 101)
X_tr, y_tr = [train.userId, train.movieId], train.rating
X_te, y_te = [test.userId, test.movieId], test.rating

Instantiate a ``MatrixFacto``  algorithm object with specified hyperparameters:

In [ ]:
latent_dim = 10
MF_model = MatrixFacto(n_users, n_movies, latent_dim)
MF_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 10)     │     97,240 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 10)     │      6,100 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 10)        │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 10)        │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 1)         │          0 │ flatten_2[0][0],  │
│                     │                   │            │ flatten_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 103,340 (403.67 KB)

 Trainable params: 103,340 (403.67 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
path_checkpt = "MF_checkpt.weights.h5"
es_callback = EarlyStopping(monitor="val_rmse", patience=5)

modelckpt_callback = ModelCheckpoint(
    monitor="val_rmse",
    filepath=path_checkpt,
    verbose=1,
    save_weights_only=True,
    save_best_only=True,
)

In [ ]:
history = MF_model.fit(X_tr, y_tr, validation_data=(X_te, y_te), batch_size =128, epochs=40, callbacks=[es_callback, modelckpt_callback])

Epoch 1/40
631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 13.3076 - rmse: 3.6479
Epoch 1: val_rmse improved from None to 3.38340, saving model to MF_checkpt.weights.h5

Epoch 1: finished saving model to MF_checkpt.weights.h5
631/631 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 13.0086 - rmse: 3.6067 - val_loss: 11.4480 - val_rmse: 3.3834
Epoch 2/40
617/631 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 9.3983 - rmse: 3.0604
Epoch 2: val_rmse improved from 3.38340 to 2.17545, saving model to MF_checkpt.weights.h5

Epoch 2: finished saving model to MF_checkpt.weights.h5
631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 7.5692 - rmse: 2.7509 - val_loss: 4.7360 - val_rmse: 2.1754
Epoch 3/40
628/631 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.8636 - rmse: 1.9631
Epoch 3: val_rmse improved from 2.17545 to 1.70207, saving model to MF_checkpt.weights.h5

Epoch 3: finished saving model to MF_checkpt.weights.h5
631/631 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 3.4065 - rmse: 1.8444 - val_loss: 2.9027 - val

### Model evaluation

Load the best validated model and check the validation error

In [ ]:
MF_model.load_weights(path_checkpt)
MF_model.evaluate(X_te, y_te)

631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1.2082 - rmse: 1.0916


[1.208220362663269, 1.0915533304214478]

### Prediction

In [ ]:
df = test[test['userId'] == 0]
df

,userId,movieId,rating
204,0,204,4.0
215,0,215,5.0
83,0,83,3.0
71,0,71,5.0
131,0,131,4.0
178,0,178,5.0
16,0,16,3.0
225,0,225,5.0
170,0,170,2.0
5,0,5,3.0


In [ ]:
df = df.sort_values(by = 'movieId')
df = df.reset_index(drop=True)
df

,userId,movieId,rating
0,0,0,4.0
1,0,1,4.0
2,0,2,4.0
3,0,5,3.0
4,0,14,4.0
5,0,16,3.0
6,0,18,5.0
7,0,31,5.0
8,0,39,3.0
9,0,42,3.0


Now let's predict the ratings that User with ID 0 will give to the movies from the test set.

In [ ]:
pred = MF_model.predict([df.userId, df.movieId])
print(pred)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
[[4.858059 ]
 [3.917573 ]
 [4.7529655]
 [4.384852 ]
 [4.3268323]
 [5.0358024]
 [4.622041 ]
 [4.783062 ]
 [3.5620391]
 [4.1749115]
 [4.964128 ]
 [5.0641055]
 [4.306067 ]
 [4.242025 ]
 [4.525017 ]
 [5.0086436]
 [5.176332 ]
 [4.601765 ]
 [5.0006747]
 [5.018736 ]
 [4.4705443]
 [4.423999 ]
 [4.296538 ]
 [4.508616 ]
 [3.937603 ]
 [2.5243945]
 [3.6373668]
 [4.805134 ]
 [4.55998  ]
 [3.9875607]
 [4.3015456]
 [3.7981963]
 [4.114626 ]
 [3.951279 ]
 [4.4506054]
 [4.9297223]
 [4.976983 ]
 [4.9742126]
 [4.634231 ]
 [4.019106 ]
 [4.0023427]
 [4.143206 ]
 [4.3654313]
 [4.4750214]
 [4.8818283]
 [4.27744  ]
 [4.29173  ]]


In [ ]:
pd.DataFrame(pred, columns = ['rating'])

,rating
0,4.858059
1,3.917573
2,4.752965
3,4.384852
4,4.326832
5,5.035802
6,4.622041
7,4.783062
8,3.562039
9,4.174911


In [ ]:
df_pred = pd.concat([df, pd.DataFrame(pred, columns = ['pred_rating'])], axis = 1)
df_pred

,userId,movieId,rating,pred_rating
0,0,0,4.0,4.858059
1,0,1,4.0,3.917573
2,0,2,4.0,4.752965
3,0,5,3.0,4.384852
4,0,14,4.0,4.326832
5,0,16,3.0,5.035802
6,0,18,5.0,4.622041
7,0,31,5.0,4.783062
8,0,39,3.0,3.562039
9,0,42,3.0,4.174911


# Neural Collaborative Filtering

In this section, we instroduce a new way to do collaborative filtering with deep learning techiniques. There's a paper from 2017, titled [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031.pdf), which describes the approach to perform collaborative filtering using neural networks.


This a fused model with two major components that contribute to the predicted rating: matrix factorization and single-layer perceptron (a dense layer), whose outputs are combined through concatenation into the last hidden dense layer:

$$
\hat{r}_{i,j} = \sigma\left(v^\top \begin{bmatrix} p_{i}^\top q_{j} \\  \sigma\left(W\begin{bmatrix}p_{i}\\q_{j}\end{bmatrix}+ b\right)\end{bmatrix}+ c\right)
$$

where $\sigma$ is a nonlinear activation fuction, e.g. ReLU.

For the impelementation, we use the Keras **functional API**. The latent user/item vectors $p_i$'s, $q_j$'s are trainable parameters from two Embedding layers. $W, v$ are the weights from dense layers, whereas $b, c$ are the bias terms repectively.


In [ ]:
from keras.layers import concatenate, Dense

In [ ]:
def NeuralMatFacto(n_movie, n_user, latent_dim):
  # Define inputs
    movie_input = Input(shape=[1])
    user_input = Input(shape=[1])

    # Embeddings
    movie_embedding = Embedding(n_movies, latent_dim)(movie_input)
    movie_vec = Flatten()(movie_embedding)

    user_embedding = Embedding(n_users, latent_dim)(user_input)
    user_vec = Flatten()(user_embedding)

    # Dense Layer
    concat = concatenate([movie_vec, user_vec])
    dense = Dense(100, activation='relu')(concat)

    # MF Layer
    mf = dot([movie_vec, user_vec], axes=1)

    # Concatenation
    combine_mlp_mf = concatenate([mf, dense])

    # Output
    rating = Dense(1, activation='relu')(combine_mlp_mf)

    model = Model([user_input, movie_input], rating)
    model.compile(loss = 'mse', optimizer = 'adam', metrics = [tf.keras.metrics.RootMeanSquaredError(name='rmse')])

    return model

In [ ]:
latent_dim = 10
NMF_model = NeuralMatFacto(n_users, n_movies, latent_dim)
NMF_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 1, 10)     │     97,240 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 1, 10)     │      6,100 │ input_layer_11[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_12          │ (None, 10)        │          0 │ embedding_12[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_13          │ (None, 10)        │          0 │ embedding_13[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 20)        │          0 │ flatten_12[0][0], │
│ (Concatenate)       │                   │            │ flatten_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_5 (Dot)         │ (None, 1)         │          0 │ flatten_12[0][0], │
│                     │                   │            │ flatten_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 100)       │      2,100 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_7       │ (None, 101)       │          0 │ dot_5[0][0],      │
│ (Concatenate)       │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │        102 │ concatenate_7[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 105,542 (412.27 KB)

 Trainable params: 105,542 (412.27 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
path_checkpt = "NMF_checkpt.weights.h5"
es_callback = EarlyStopping(monitor="val_rmse", patience=5)

modelckpt_callback = ModelCheckpoint(
    monitor="val_rmse",
    filepath=path_checkpt,
    verbose=1,
    save_weights_only=True,
    save_best_only=True,
)

In [ ]:
history = NMF_model.fit(X_tr, y_tr, validation_data=(X_te, y_te), batch_size =128, epochs=40, callbacks=[es_callback, modelckpt_callback])

Epoch 1/40
631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.9090 - rmse: 2.1179
Epoch 1: val_rmse improved from None to 0.88656, saving model to NMF_checkpt.weights.h5

Epoch 1: finished saving model to NMF_checkpt.weights.h5
631/631 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 2.1378 - rmse: 1.4621 - val_loss: 0.7860 - val_rmse: 0.8866
Epoch 2/40
628/631 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7091 - rmse: 0.8421
Epoch 2: val_rmse improved from 0.88656 to 0.87712, saving model to NMF_checkpt.weights.h5

Epoch 2: finished saving model to NMF_checkpt.weights.h5
631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.7239 - rmse: 0.8508 - val_loss: 0.7693 - val_rmse: 0.8771
Epoch 3/40
621/631 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6733 - rmse: 0.8205
Epoch 3: val_rmse improved from 0.87712 to 0.87346, saving model to NMF_checkpt.weights.h5

Epoch 3: finished saving model to NMF_checkpt.weights.h5
631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.6881 - rmse: 0.8295 - val_loss: 0.7629 - 

In [ ]:
NMF_model.load_weights(path_checkpt)
NMF_model.evaluate(X_te, y_te)

631/631 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.7477 - rmse: 0.8647


[0.7476788759231567, 0.8646842837333679]

In [ ]:
pred = NMF_model.predict([df.userId, df.movieId])

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 350ms/step


In [ ]:
df_pred = pd.concat([df, pd.DataFrame(pred, columns = ['pred_rating'])], axis = 1)
df_pred

,userId,movieId,rating,pred_rating
0,0,0,4.0,4.733119
1,0,1,4.0,4.171699
2,0,2,4.0,4.558689
3,0,5,3.0,4.405311
4,0,14,4.0,4.256741
5,0,16,3.0,4.754357
6,0,18,5.0,4.559861
7,0,31,5.0,4.487778
8,0,39,3.0,3.925529
9,0,42,3.0,4.365353
